# Generate Spectral Datasets with Surrogate Models

In [ ]:
import os
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from rich.progress import track
from torch.utils.data import DataLoader, TensorDataset

from mcmlnet.experiments.data_loaders.config import DataConfig
from mcmlnet.experiments.data_loaders.simulation import (
    GenericSimulationLoader,
    LanSimulationLoader,
    ManojlovicSimulationLoader,
    TsuiSimulationLoader,
)
from mcmlnet.experiments.data_loaders.utils import subsample_data
from mcmlnet.experiments.plotting import setup_plot_style
from mcmlnet.training.data_loading.sota_data_classes import (
    SOTAPreprocessor,
)
from mcmlnet.training.models.base_model import BaseModel
from mcmlnet.transforms.related_work_transformations import (
    LanConstants,
    LanTransformation,
    ManojlovicConstants,
    TsuiConstants,
    TsuiTransformation,
)
from mcmlnet.utils.convenience import (
    load_trained_model,
)
from mcmlnet.utils.load_configs import (
    load_config,
)
from mcmlnet.utils.metrics import mape, nmae, nrmse

load_dotenv()

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
if not os.path.exists(os.environ["plot_save_dir"]):
    os.makedirs(os.environ["plot_save_dir"])

## Load Data

**Note**: A lot of papers are defined only for narrower wavelength ranges:

- **Tsui 2018**: 410-760 nm
    - Nunez's dissertation ${\color{red}\text{cannot be the collagen source}}$, as all figures (except CIE color space representations) do not extend below 450 nm. Therefore, simulations should begin at 460 nm.
    
- **Wang 2019**: 401-583 nm 
    - This range may be too narrow for comprehensive analysis. Additionally, there appears to be an erroneous link to Buiteveld's water data in Figure 2, and the collagen wavelength range issue is the same as for Tsui 2018.
    
- **Lan 2023**: 450-650 nm 
    - Uses simplified phantoms consisting of "monodisperse polystyrene microsphere suspension as scatterer and hemoglobin powder dissolved in water as absorber," with hemoglobin as the sole absorbing component.
    
- **Manojlovic 2025**: 400-1000 nm 
    - Although initially covering a broad range, the effective wavelength range was restricted in their analysis to 430-750 nm.

In [ ]:
# load experiment config for network paths
cfg = load_config()

# load related work data
SAVE_DIR = os.path.join(os.getenv("data_dir"), "raw/related_work_reimplemented")
RESULT_DIR = os.path.join(os.getenv("data_dir"), "raw/inference")
data_lan = pd.read_parquet(os.path.join(SAVE_DIR, "lan_lhs_2023_resim.parquet")).drop(
    columns=["layer [top first]"]
)
data_tsui = pd.read_parquet(os.path.join(SAVE_DIR, "tsui_2018_resim.parquet")).drop(
    columns=["layer [top first]"]
)
print(f"Data Lan shape: {data_lan.shape}")
print(f"Data Tsui shape: {data_tsui.shape}")


def sample_uniformly_with_seed(
    low: float,
    high: float,
    n_samples: int,
    seed: int = 0,
) -> torch.Tensor:
    """Sample uniformly from a given range with a fixed seed

    Args:
        low: The lower bound of the range.
        high: The upper bound of the range.
        n_samples: The number of samples to generate.
        seed: The seed to use for the random number generator.

    Returns:
        A tensor of shape (n_samples,) containing the sampled values.
    """
    torch.manual_seed(seed)
    return torch.rand(n_samples) * (high - low) + low

## Workflow Overview

1. **Define transformation functions and wavelength ranges** - Establish the mappings between physiological and physical parameters, along with the spectral ranges for each state-of-the-art (SOTA) simulation database.

2. **Sample physiological parameters** - Generate 100,000 physiological parameter samples for individual SOTA simulation databases, ensuring coverage within the specified inference ranges established by each publication.

3. **Compute reflectance spectra** - Apply the trained neural network surrogate models to regress reflectance values for the sampled parameter sets.

4. **Evaluate surrogate model performance** - Calculate test metrics on the held-out test split of the training data to assess the performance and compare to reported values.

5. **Archive results for downstream analysis** - Save the generated reflectance spectra and associated metadata for use in subsequent experiments and comparative studies.

## Lan et al. 2023

### Define Physiological to Physical Parameter Transformation

In [ ]:
data_lan.describe()

In [ ]:
consts_lan = LanConstants()


def generate_physical_data_lan(
    sao2: torch.Tensor,
    vhb: torch.Tensor,
    a_mie: torch.Tensor,
    b_mie: torch.Tensor,
    a_ray: torch.Tensor,
    g: torch.Tensor,
    transformation: LanTransformation,
    wavelengths: torch.Tensor,
) -> torch.Tensor:
    """
    Generate the physical parameters for the single tissue layer Lan model (2023).

    Args:
        sao2: The oxygen saturation.
        vhb: The hemoglobin concentration.
        a_mie: The absorption coefficient of the Mie scattering.
        b_mie: The scattering coefficient of the Mie scattering.
        a_ray: The absorption coefficient of the Rayleigh scattering.
        g: The anisotropy factor.
        transformation: The transformation functions for the physical parameters.
        wavelengths: The wavelengths to consider.

    Returns:
        The physical parameters for a single layer.
    """

    # compute and stack the physical parameters
    physical_params = torch.stack(
        [
            transformation.absorption(sao2, vhb),
            transformation.scattering(a_mie, b_mie, a_ray, g),
            g.reshape(-1, 1).repeat(1, len(wavelengths)),
            torch.ones_like(sao2).reshape(-1, 1).repeat(1, len(wavelengths))
            * consts_lan.n,
            torch.ones_like(sao2).reshape(-1, 1).repeat(1, len(wavelengths))
            * 0.2,  # thick deepest layer
        ],
        dim=-1,
    )
    return physical_params


def load_lan_model() -> tuple[BaseModel, Any, Any]:
    """
    Load the trained Lan model.

    Returns:
        The trained Lan model.
    """
    base_path = os.path.join(
        os.environ["data_dir"], cfg["surrogate"]["lan_model"]["base_path"]
    )
    checkpoint_path = cfg["surrogate"]["lan_model"]["checkpoint_path"]
    return load_trained_model(base_path, checkpoint_path, BaseModel)


def lan_model_forward(
    physical_params: torch.Tensor, batch_size: int = 1000
) -> torch.Tensor:
    """
    Compute the reflectance spectra for the given physical parameters.

    Args:
        physical_params: The physical parameters to consider.
        batch_size: The batch size to use for the model inference.

    Returns:
        The reflectance spectra.
    """
    # load the model
    model, preprocessor, cfg = load_lan_model()
    physical_params = physical_params.clone().float().reshape(-1, 1, cfg.n_params)
    # pre-process the physical_params
    physical_params[..., :2] = torch.log10(physical_params[..., :2])
    physical_params[..., -1] = torch.log10(physical_params[..., -1])
    data = preprocessor(
        torch.cat(
            [
                physical_params,
                torch.zeros((physical_params.shape[0], physical_params.shape[1], 1)),
            ],
            dim=-1,
        )
    )
    if cfg.dataset.thick_deepest_layer:
        # set thickness of the deepest layer to zero to avoid artifacts
        data[..., -2] = 0

    # Create a TensorDataset and DataLoader
    dataset = TensorDataset(data[..., :-1])
    dataloader = DataLoader(dataset, batch_size=len(data) // 10, shuffle=False)

    # Run predictions in batches
    all_predictions = []
    model.eval().cuda()
    with torch.no_grad():
        for batch in track(dataloader, description="Predicting reflectance spectra"):
            inputs = batch[0]
            predictions = model(inputs.reshape(-1, cfg.n_params).cuda()).cpu()
            all_predictions.append(predictions)

    return torch.cat(all_predictions, dim=0).detach().cpu()

### Sample and Filter Physical Parameters

In [ ]:
# define the transformation functions
wavelengths_lan = torch.from_numpy(consts_lan.wavelengths * 1e9)  # in nm
lan_physical = LanTransformation(wavelengths_lan)
print(f"Wavelengths Lan: {wavelengths_lan.tolist()}")

# sample physiological parameters
n_samples = consts_lan.n_samples_surrogate
physio_params = {
    "sao2": [],
    "vhb": [],
    "a_mie": [],
    "b_mie": [],
    "a_ray": [],
    "g": [],
}
counter = 0

# repeatedly sample until threshold is reached
while len(physio_params["sao2"]) < n_samples:
    # sample uniformly from a very broad physiological range
    sao2 = sample_uniformly_with_seed(
        consts_lan.sao2_range[0], consts_lan.sao2_range[1], n_samples, seed=0 + counter
    )
    vhb = sample_uniformly_with_seed(
        consts_lan.vhb_range[0], consts_lan.vhb_range[1], n_samples, seed=1 + counter
    )
    a_mie = sample_uniformly_with_seed(
        consts_lan.a_mie_range[0],
        consts_lan.a_mie_range[1],
        n_samples,
        seed=2 + counter,
    )
    b_mie = sample_uniformly_with_seed(
        consts_lan.b_mie_range[0],
        consts_lan.b_mie_range[1],
        n_samples,
        seed=3 + counter,
    )
    a_ray = torch.zeros_like(sao2)
    g = sample_uniformly_with_seed(
        consts_lan.g_range[0], consts_lan.g_range[1], n_samples, seed=5 + counter
    )

    # generate physical parameters
    physical_params = generate_physical_data_lan(
        sao2, vhb, a_mie, b_mie, a_ray, g, lan_physical, wavelengths_lan
    )
    # remove all parameter rows that are not in the covered simulation
    mask = (
        physical_params[..., :-2] >= torch.from_numpy(data_lan.min(axis=0).values)[:-3]
    ) & (
        physical_params[..., :-2] <= torch.from_numpy(data_lan.max(axis=0).values)[:-3]
    )
    mask = mask.all(dim=-1).all(dim=-1)

    # append the sampled physiological parameters
    physio_params["sao2"] += sao2[mask].tolist()
    physio_params["vhb"] += vhb[mask].tolist()
    physio_params["a_mie"] += a_mie[mask].tolist()
    physio_params["b_mie"] += b_mie[mask].tolist()
    physio_params["a_ray"] += a_ray[mask].tolist()
    physio_params["g"] += g[mask].tolist()

    counter += 1
    if counter % 100 == 0:
        print(
            f"Sampled {len(physio_params['sao2'])}/{n_samples} "
            "physiological parameter sets."
        )

# concatenate the sampled physiological parameters
sao2 = torch.tensor(physio_params["sao2"])[:n_samples]
vhb = torch.tensor(physio_params["vhb"])[:n_samples]
a_mie = torch.tensor(physio_params["a_mie"])[:n_samples]
b_mie = torch.tensor(physio_params["b_mie"])[:n_samples]
a_ray = torch.tensor(physio_params["a_ray"])[:n_samples]
g = torch.tensor(physio_params["g"])[:n_samples]
print(f"SaO2 shape: {sao2.shape}")

# save the sampled physiological parameters as a DataFrame
physio_params_df = pd.DataFrame(
    {
        "sao2": sao2.numpy(),
        "vhb": vhb.numpy(),
        "a_mie": a_mie.numpy(),
        "b_mie": b_mie.numpy(),
        "a_ray": a_ray.numpy(),
        "g": g.numpy(),
    }
)
print(physio_params_df.head())
physio_params_df.to_csv(
    os.path.join(
        RESULT_DIR,
        "physio_params_lan.csv",
    )
)
# test load physio_params_df
physio_params_df = pd.read_csv(
    os.path.join(
        RESULT_DIR,
        "physio_params_lan.csv",
    ),
    index_col=0,
)
print(physio_params_df.head())

In [ ]:
# generate physical parameters
physical_params = generate_physical_data_lan(
    sao2, vhb, a_mie, b_mie, a_ray, g, lan_physical, wavelengths_lan
)

# check the physical parameter ranges
print(physical_params.shape)
print(physical_params.min(dim=0).values.min(dim=0).values)
print(physical_params.max(dim=0).values.max(dim=0).values)
# check that there are no duplicates
unique_params = torch.unique(physical_params, dim=0)
print(f"Unique params: {len(unique_params)}")

# plot the sampled physiological parameter distributions as histograms
setup_plot_style()

label_mapping = {
    "sao2": r"s$_t$O$_2 [\%]$",
    "vhb": r"$v_{Hb}$",
    "a_mie": r"$a_{Mie} [cm^{-1}]$",
    "b_mie": r"$b_{Mie}$",
    "a_ray": r"$a_{Ray}$",
    "g": r"$g$",
}

fig, axes = plt.subplots(2, 3, figsize=(6.5, 4))
axes = axes.flatten()
for i, col in enumerate(physio_params_df.columns):
    if col == "sao2":
        # scale sao2 to percentage
        _vals = physio_params_df[col].to_numpy() * 100
    elif col == "a_mie":
        # scale a_mie to cm^-1
        _vals = physio_params_df[col].to_numpy() * 0.01
    else:
        _vals = physio_params_df[col].to_numpy()
    axes[i].hist(_vals, bins=200)
    axes[i].set_title(label_mapping[col])
    axes[i].grid(True, linestyle="--", linewidth=0.5)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")
fig.suptitle(
    "Sampled Physiological Parameters for Inference with Lan et al. (2023)", fontsize=12
)
plt.subplots_adjust(top=0.85, hspace=0.5, wspace=0.5)
plt.savefig(
    os.path.join(
        os.getenv("plot_save_dir"),
        "inference_marginals_physio_params_lan.svg",
    )
)
plt.show()

### Compute Test Metrics

In [ ]:
# load the test data
_data = LanSimulationLoader(specular=False).load_data(lhs=True).to_numpy()
split = SOTAPreprocessor("lan_lhs", 1, False, None)
lan_test = _data[split.consistent_data_split_ids(_data, mode="test")]
lan_test_params = torch.from_numpy(lan_test[:, :-1])
lan_test_reflectances = torch.from_numpy(
    lan_test[:, -1].reshape(-1, 1),
)
# run surrogate model on the test data
lan_test_pred = lan_model_forward(lan_test_params, batch_size=1000)

# comptue the MAPE, NMAE, and NRMSE
print(f"MAPE: {mape(lan_test_pred, lan_test_reflectances)}")
print(f"NMAE: {nmae(lan_test_pred, lan_test_reflectances)}")
print(f"NRMSE: {nrmse(lan_test_pred, lan_test_reflectances)}")

### Run Forward Model of Physical Parameters for SOTA Comparison

In [ ]:
reflectances_lan = lan_model_forward(physical_params).reshape(-1, len(wavelengths_lan))
print(f"Reflectances Lan shape: {reflectances_lan.shape}")

In [ ]:
# plot example reflectance spectra
n_samples = 5
plt.figure(figsize=(10, 5))
for i in range(n_samples):
    plt.plot(
        wavelengths_lan,
        reflectances_lan.reshape(-1, len(wavelengths_lan))[i].numpy(),
        label=f"Sample {i}",
    )
plt.xlabel("Wavelength [nm]")
plt.ylabel("Reflectance")
plt.title("Example Reflectance Spectra (Lan et al. 2023)")
plt.grid()
plt.legend()
plt.show()

In [ ]:
# save the reflectance spectra for PCA coverage comparison
np.savez_compressed(
    os.path.join(
        RESULT_DIR,
        "reflectances_lan.npz",
    ),
    reflectances_lan=reflectances_lan.numpy(),
)
# save the physical parameters for PCA coverage comparison
np.savez_compressed(
    os.path.join(
        RESULT_DIR,
        "physical_params_lan.npz",
    ),
    physical_params=physical_params.numpy(),
)

In [ ]:
# test loading the reflectance spectra and physical parameters
reflectances_lan = np.load(
    os.path.join(
        RESULT_DIR,
        "reflectances_lan.npz",
    )
)["reflectances_lan"]
physical_params = np.load(
    os.path.join(
        RESULT_DIR,
        "physical_params_lan.npz",
    )
)["physical_params"]
print(f"Reflectances Lan shape: {reflectances_lan.shape}")
print(f"Physical params Lan shape: {physical_params.shape}")

## Tsui et al. 2018

### Define Physiological to Physical Parameter Transformation

In [ ]:
data_tsui.describe()

In [ ]:
consts_tsui = TsuiConstants()


def generate_physical_data_tsui(
    a_1: torch.Tensor,
    b_1: torch.Tensor,
    a_2: torch.Tensor,
    b_2: torch.Tensor,
    a_4: torch.Tensor,
    b_4: torch.Tensor,
    f_m: torch.Tensor,
    f_b: torch.Tensor,
    sao2: torch.Tensor,
    d_1: torch.Tensor,
    d_2: torch.Tensor,
    d_3: torch.Tensor,
    transformation: TsuiTransformation,
    wavelengths: torch.Tensor,
) -> torch.Tensor:
    """
    Generate the physical parameters for the single tissue layer Tsui model (2018).

    Args:
        a_1: The scattering amplitude a_mie of the first layer.
        b_1: The scattering power b_mie of the first layer.
        a_2: The scattering amplitude a_mie of the second layer.
        b_2: The scattering power b_mie of the second layer.
        a_4: The scattering amplitude a_mie of the fourth layer.
        b_4: The scattering power b_mie of the fourth layer.
        f_m: The melanin fraction.
        f_b: The blood volume fraction.
        sao2: The oxygen saturation.
        d_1: The thickness of the first layer.
        d_2: The thickness of the second layer.
        d_3: The thickness of the third layer.
        transformation: The transformation functions for the physical parameters.
        wavelengths: The wavelengths to consider.

    Returns:
        The physical parameters for a single layer.
    """
    # compute mu_a_1_and_2
    mu_a_1_and_2 = transformation.mu_a_1_and_2().repeat(n_samples, 1)
    mu_s2 = transformation.scattering(a_2, b_2, torch.Tensor([0.75]))

    # compute physical parameters
    physical_params = torch.stack(
        [
            mu_a_1_and_2,
            transformation.scattering(a_1, b_1, torch.Tensor([consts_tsui.g_1])),
            torch.ones_like(mu_a_1_and_2) * consts_tsui.g_1,
            torch.ones_like(mu_a_1_and_2) * consts_tsui.n,
            d_1.reshape(-1, 1).repeat(1, len(wavelengths)),
            mu_a_1_and_2,
            mu_s2,
            torch.ones_like(mu_a_1_and_2) * consts_tsui.g_2,
            torch.ones_like(mu_a_1_and_2) * consts_tsui.n,
            d_2.reshape(-1, 1).repeat(1, len(wavelengths)),
            transformation.mu_a_3(f_m),
            mu_s2 * 1.35,
            torch.ones_like(mu_a_1_and_2) * consts_tsui.g_3,
            torch.ones_like(mu_a_1_and_2) * consts_tsui.n,
            d_3.reshape(-1, 1).repeat(1, len(wavelengths)),
            transformation.mu_a_4(
                f_b, sao2, torch.Tensor([consts_tsui.f_w] * f_b.shape[0])
            ),
            transformation.scattering(a_4, b_4, torch.Tensor([consts_tsui.g_4])),
            torch.ones_like(mu_a_1_and_2) * consts_tsui.g_4,
            torch.ones_like(mu_a_1_and_2) * consts_tsui.n,
            torch.ones_like(mu_a_1_and_2) * 0.2,
        ],
        dim=-1,
    )

    return physical_params


def load_tsui_model() -> tuple[BaseModel, Any, Any]:
    """
    Load the trained Tsui model.

    Returns:
        The trained Tsui model.
    """
    base_path = os.path.join(
        os.environ["data_dir"], cfg["surrogate"]["tsui_model"]["base_path"]
    )
    checkpoint_path = cfg["surrogate"]["tsui_model"]["checkpoint_path"]
    return load_trained_model(base_path, checkpoint_path, BaseModel)


def tsui_model_forward(
    physical_params: torch.Tensor, batch_size: int = 1000
) -> torch.Tensor:
    """
    Compute the reflectance spectra for the given physical parameters.

    Args:
        physical_params: The physical parameters to consider.
        preprocessor: The preprocessor to use for the model inference.
        batch_size: The batch size to use for the model inference.

    Returns:
        The reflectance spectra.
    """
    # load the model
    model, preprocessor, cfg = load_tsui_model()
    physical_params = physical_params.clone().float().reshape(-1, 1, cfg.n_params)
    # pre-process the physical_params
    log_indices = [
        slice(0, 2),
        slice(5, 7),
        slice(10, 12),
        slice(15, 17),
        [4, 9, 14, 19],
    ]

    for idx in log_indices:
        physical_params[..., idx] = torch.log10(physical_params[..., idx])
    data = preprocessor(
        torch.cat(
            [
                physical_params,
                torch.zeros((physical_params.shape[0], physical_params.shape[1], 1)),
            ],
            dim=-1,
        )
    )
    if cfg.dataset.thick_deepest_layer:
        # set thickness of the deepest layer to zero to avoid artifacts
        data[..., -2] = 0

    # Create a TensorDataset and DataLoader
    dataset = TensorDataset(data[..., :-1])
    dataloader = DataLoader(dataset, batch_size=len(data) // 10, shuffle=False)

    # Run predictions in batches
    all_predictions = []
    model.eval().cuda()
    with torch.no_grad():
        for batch in track(dataloader, description="Predicting reflectance spectra"):
            inputs = batch[0]
            predictions = model(inputs.reshape(-1, cfg.n_params).cuda()).cpu()
            all_predictions.append(predictions)

    return torch.cat(all_predictions, dim=0).detach().cpu()

### Sample and Filter Physical Parameters

In [ ]:
# define the transformation functions
wavelengths_tsui = torch.from_numpy(consts_tsui.wavelengths * 1e9)  # in nm
tsui_physical = TsuiTransformation(wavelengths_tsui)
print(f"Wavelengths Tsui: {wavelengths_tsui.tolist()}")

# sample physiological parameters
n_samples = consts_tsui.n_samples_surrogate
physio_params = {
    "a_1": [],
    "b_1": [],
    "a_2": [],
    "b_2": [],
    "a_4": [],
    "b_4": [],
    "f_m": [],
    "f_b": [],
    "sao2": [],
    "d_1": [],
    "d_2": [],
    "d_3": [],
}
counter = 0

# repeatedly sample until threshold is reached
while len(physio_params["sao2"]) < n_samples:
    # sample uniformly from the stated ranges in the paper
    a_1 = sample_uniformly_with_seed(
        consts_tsui.a_1_range[0], consts_tsui.a_1_range[1], n_samples, 0 + counter
    )
    b_1 = sample_uniformly_with_seed(
        consts_tsui.b_1_range[0], consts_tsui.b_1_range[1], n_samples, 1 + counter
    )
    a_2 = sample_uniformly_with_seed(
        consts_tsui.a_2_range[0], consts_tsui.a_2_range[1], n_samples, 2 + counter
    )
    b_2 = sample_uniformly_with_seed(
        consts_tsui.b_2_range[0], consts_tsui.b_2_range[1], n_samples, 3 + counter
    )
    a_4 = sample_uniformly_with_seed(
        consts_tsui.a_4_range[0], consts_tsui.a_4_range[1], n_samples, 4 + counter
    )
    b_4 = sample_uniformly_with_seed(
        consts_tsui.b_4_range[0], consts_tsui.b_4_range[1], n_samples, 5 + counter
    )
    f_m = sample_uniformly_with_seed(
        consts_tsui.f_m_range[0], consts_tsui.f_m_range[1], n_samples, 6 + counter
    )
    f_b = sample_uniformly_with_seed(
        consts_tsui.f_b_range[0], consts_tsui.f_b_range[1], n_samples, 7 + counter
    )
    sao2 = sample_uniformly_with_seed(
        consts_tsui.sao2_range[0], consts_tsui.sao2_range[1], n_samples, 8 + counter
    )

    # additional thickness parameters
    d_1 = sample_uniformly_with_seed(
        consts_tsui.d_1_range[0], consts_tsui.d_1_range[1], n_samples, 9 + counter
    )
    d_2 = sample_uniformly_with_seed(
        consts_tsui.d_2_range[0], consts_tsui.d_2_range[1], n_samples, 10 + counter
    )
    d_3 = sample_uniformly_with_seed(
        consts_tsui.d_3_range[0], consts_tsui.d_3_range[1], n_samples, 11 + counter
    )

    # generate physical parameters
    physical_params = generate_physical_data_tsui(
        a_1,
        b_1,
        a_2,
        b_2,
        a_4,
        b_4,
        f_m,
        f_b,
        sao2,
        d_1,
        d_2,
        d_3,
        tsui_physical,
        wavelengths_tsui,
    )

    # remove all parameter rows that are not in the covered simulation range
    mask = (
        physical_params >= torch.from_numpy(data_tsui.min(axis=0).to_numpy())[:-1]
    ) & (physical_params <= torch.from_numpy(data_tsui.max(axis=0).to_numpy())[:-1])
    mask = mask.all(dim=-1).all(dim=-1)

    # append the sampled physiological parameters
    physio_params["a_1"] += a_1[mask].tolist()
    physio_params["b_1"] += b_1[mask].tolist()
    physio_params["a_2"] += a_2[mask].tolist()
    physio_params["b_2"] += b_2[mask].tolist()
    physio_params["a_4"] += a_4[mask].tolist()
    physio_params["b_4"] += b_4[mask].tolist()
    physio_params["f_m"] += f_m[mask].tolist()
    physio_params["f_b"] += f_b[mask].tolist()
    physio_params["sao2"] += sao2[mask].tolist()
    physio_params["d_1"] += d_1[mask].tolist()
    physio_params["d_2"] += d_2[mask].tolist()
    physio_params["d_3"] += d_3[mask].tolist()

    counter += 1
    if counter % 100 == 0:
        print(
            f"Sampled {len(physio_params['sao2'])}/{n_samples} "
            "physiological parameter sets."
        )

# concatenate the sampled physiological parameters
a_1 = torch.tensor(physio_params["a_1"])[:n_samples]
b_1 = torch.tensor(physio_params["b_1"])[:n_samples]
a_2 = torch.tensor(physio_params["a_2"])[:n_samples]
b_2 = torch.tensor(physio_params["b_2"])[:n_samples]
a_4 = torch.tensor(physio_params["a_4"])[:n_samples]
b_4 = torch.tensor(physio_params["b_4"])[:n_samples]
f_m = torch.tensor(physio_params["f_m"])[:n_samples]
f_b = torch.tensor(physio_params["f_b"])[:n_samples]
sao2 = torch.tensor(physio_params["sao2"])[:n_samples]
d_1 = torch.tensor(physio_params["d_1"])[:n_samples]
d_2 = torch.tensor(physio_params["d_2"])[:n_samples]
d_3 = torch.tensor(physio_params["d_3"])[:n_samples]
print(f"SaO2 shape: {sao2.shape}")

# save the sampled physiological parameters as a DataFrame
physio_params_df = pd.DataFrame(
    {
        "a_1": a_1.numpy(),
        "b_1": b_1.numpy(),
        "a_2": a_2.numpy(),
        "b_2": b_2.numpy(),
        "a_4": a_4.numpy(),
        "b_4": b_4.numpy(),
        "f_m": f_m.numpy(),
        "f_b": f_b.numpy(),
        "sao2": sao2.numpy(),
        "d_1": d_1.numpy(),
        "d_2": d_2.numpy(),
        "d_3": d_3.numpy(),
    }
)
print(physio_params_df.head())
physio_params_df.to_csv(
    os.path.join(
        RESULT_DIR,
        "physio_params_tsui.csv",
    )
)
# test load physio_params_df
physio_params_df = pd.read_csv(
    os.path.join(
        RESULT_DIR,
        "physio_params_tsui.csv",
    ),
    index_col=0,
)
print(physio_params_df.head())

In [ ]:
# generate physical parameters
physical_params = generate_physical_data_tsui(
    a_1,
    b_1,
    a_2,
    b_2,
    a_4,
    b_4,
    f_m,
    f_b,
    sao2,
    d_1,
    d_2,
    d_3,
    tsui_physical,
    wavelengths_tsui,
)

# check the physical parameter ranges
print(f"Physical params shape: {physical_params.shape}")
print(f"Physical params min: {physical_params.min(dim=0).values.min(dim=0).values}")
print(f"Physical params max: {physical_params.max(dim=0).values.max(dim=0).values}")
# check that there are no duplicates
unique_params = torch.unique(physical_params, dim=0)
print(f"Unique params: {len(unique_params)}")

# plot the sampled physiological parameter distributions as histograms
setup_plot_style()

label_mapping = {
    "a_1": r"$C_{Mie,1} [a.u.]$",
    "b_1": r"$b_{Mie,1}$",
    "a_2": r"$C_{Mie,2} [a.u.]$",
    "b_2": r"$b_{Mie,2}$",
    "a_4": r"$C_{Mie,4} [a.u.]$",
    "b_4": r"$b_{Mie,4}$",
    "f_m": r"$f_{mel.}$",
    "f_b": r"$v_{Hb}$",
    "sao2": r"s$_t$O$_2 [\%]$",
    "d_1": r"$d_{1} [\mu m]$",
    "d_2": r"$d_{2} [\mu m]$",
    "d_3": r"$d_{3} [\mu m]$",
}

fig, axes = plt.subplots(3, 4, figsize=(6.5, 5))
axes = axes.flatten()
for i, col in enumerate(physio_params_df.columns):
    if col == "sao2":
        # scale sao2 to percentage
        axes[i].hist(physio_params_df[col].to_numpy() * 100, bins=200)
    elif col in ["d_1", "d_2", "d_3"]:
        # scale d_1, d_2, d_3 to µm
        axes[i].hist(physio_params_df[col].to_numpy() * 10**6, bins=200)
    else:
        axes[i].hist(physio_params_df[col].to_numpy(), bins=200)
    axes[i].set_title(label_mapping[col])
    axes[i].grid(True, linestyle="--", linewidth=0.5)
    # Manually manipulate x-axis ticks for better readability
    if col == "sao2":
        x_ticks = [0, 50, 100]
        axes[i].set_xticks(x_ticks)
        axes[i].set_xticklabels([f"{tick}" for tick in x_ticks])
    elif col in ["d_1", "d_2"]:
        x_ticks = [0, 10, 20, 30]
        axes[i].set_xticks(x_ticks)
        axes[i].set_xticklabels([f"{tick}" for tick in x_ticks])
    elif col == "d_3":
        x_ticks = [10, 35, 60]
        axes[i].set_xticks(x_ticks)
        axes[i].set_xticklabels([f"{tick}" for tick in x_ticks])
    elif col in ["b_1", "b_2", "b_4"]:
        x_ticks = [1.0, 1.5, 2.0]
        axes[i].set_xticks(x_ticks)
        axes[i].set_xticklabels([f"{tick:.1f}" for tick in x_ticks])
    elif col == "f_mel":
        x_ticks = [0.0, 0.1, 0.2, 0.3]
        axes[i].set_xticks(x_ticks)
        axes[i].set_xticklabels([f"{tick:.1f}" for tick in x_ticks])
    elif col in ["a_1", "a_2", "a_4"]:
        x_ticks = [0.0, 2.5e8, 5e8]
        axes[i].set_xticks(x_ticks)
        axes[i].set_xticklabels([f"{tick:.1e}" for tick in x_ticks])
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")
fig.suptitle(
    "Sampled Physiological Parameters for Inference with Tsui et al. (2018)",
    fontsize=12,
)
plt.subplots_adjust(top=0.9, hspace=0.7, wspace=0.6)
plt.savefig(
    os.path.join(
        os.getenv("plot_save_dir"),
        "inference_marginals_physio_params_tsui.svg",
    )
)
plt.show()

Note: Surprisingly many (>95%) of the sampled parameters lie outside of the domain of generated physical simulations ... How did the authors handle that ${\color{red}\text{low sample efficiency}}$...?

### Compute Test Metrics

In [ ]:
# load the test data
_data = TsuiSimulationLoader(specular=False).load_data().to_numpy()
split = SOTAPreprocessor("tsui", 1, False, None)
tsui_test = _data[split.consistent_data_split_ids(_data, mode="test")]
tsui_test_params = torch.from_numpy(tsui_test[:, :-1])
tsui_test_reflectances = torch.from_numpy(
    tsui_test[:, -1].reshape(-1, 1),
)
# run surrogate model on the test data
tsui_test_pred = tsui_model_forward(tsui_test_params, batch_size=1000)

# comptue the MAPE, NMAE, and NRMSE
print(f"MAPE: {mape(tsui_test_pred, tsui_test_reflectances)}")
print(f"NMAE: {nmae(tsui_test_pred, tsui_test_reflectances)}")
print(f"NRMSE: {nrmse(tsui_test_pred, tsui_test_reflectances)}")

### Run Forward Model of Physical Parameters for SOTA Comparison

In [ ]:
reflectances_tsui = tsui_model_forward(physical_params).reshape(
    -1, len(wavelengths_tsui)
)
print(f"Reflectances Tsui shape: {reflectances_tsui.shape}")

In [ ]:
# plot example reflectance spectra
n_samples = 5
plt.figure(figsize=(10, 5))
for i in range(n_samples):
    plt.plot(
        wavelengths_tsui,
        reflectances_tsui.reshape(-1, len(wavelengths_tsui))[i].numpy(),
        label=f"Sample {i}",
    )
plt.xlabel("Wavelength [nm]")
plt.ylabel("Reflectance")
plt.yscale("log")  # as in Fig. 3 and Fig. 4 of their paper
plt.title("Example Reflectance Spectra (Tsui et al. 2018)")
plt.grid()
plt.legend()
plt.show()

In [ ]:
# save the reflectance spectra for PCA coverage comparison
np.savez_compressed(
    os.path.join(
        RESULT_DIR,
        "reflectances_tsui.npz",
    ),
    reflectances_tsui=reflectances_tsui.numpy(),
)
np.savez_compressed(
    os.path.join(
        RESULT_DIR,
        "physical_params_tsui.npz",
    ),
    physical_params=physical_params.numpy(),
)

In [ ]:
# test loading the reflectance spectra and physical parameters
reflectances_tsui = np.load(
    os.path.join(
        RESULT_DIR,
        "reflectances_tsui.npz",
    )
)["reflectances_tsui"]
physical_params = np.load(
    os.path.join(
        RESULT_DIR,
        "physical_params_tsui.npz",
    )
)["physical_params"]
print(f"Reflectances Tsui shape: {reflectances_tsui.shape}")
print(f"Physical params shape: {physical_params.shape}")

## Manojlovic et al. 2025

### Define Physiological to Physical Parameter Transformation

In [ ]:
consts_manoj = ManojlovicConstants()
wavelengths_manoj = torch.from_numpy(consts_manoj.wavelengths * 1e9)  # in nm
print(f"Wavelengths: {wavelengths_manoj.tolist()}")


def load_manoj_model() -> tuple[BaseModel, Any, Any]:
    """
    Load the trained Manojlovic model.

    :return: The trained Manojlovic model.
    """
    base_path = os.path.join(
        os.environ["data_dir"], cfg["surrogate"]["manoj_model"]["base_path"]
    )
    checkpoint_path = cfg["surrogate"]["manoj_model"]["checkpoint_path"]
    return load_trained_model(base_path, checkpoint_path, BaseModel)


def manoj_model_forward(
    physical_params: torch.Tensor, batch_size: int = 1000
) -> torch.Tensor:
    """Compute the reflectance spectra for the given physical parameters.

    Args:
        physical_params: The physical parameters to consider.
        batch_size: The batch size to use for the predictions.

    Returns:
        The reflectance spectra.
    """
    # load the model
    model, preprocessor, cfg = load_manoj_model()
    physical_params = physical_params.clone().float().reshape(-1, 1, cfg.n_params)
    # pre-process the physical_params
    physical_params[..., :2] = torch.log10(physical_params[..., :2])
    physical_params[..., 5:7] = torch.log10(physical_params[..., 5:7])
    data = preprocessor(
        torch.cat(
            [
                physical_params,
                torch.zeros((physical_params.shape[0], physical_params.shape[1], 1)),
            ],
            dim=-1,
        )
    )
    if cfg.dataset.thick_deepest_layer:
        # set thickness of the deepest layer to zero to avoid artifacts
        data[..., -2] = 0

    # Create a TensorDataset and DataLoader
    dataset = TensorDataset(data[..., :-1])
    dataloader = DataLoader(dataset, batch_size=len(data) // 10, shuffle=False)

    # Run predictions in batches
    all_predictions = []
    model.eval().cuda()
    with torch.no_grad():
        for batch in track(dataloader, description="Predicting reflectance spectra"):
            inputs = batch[0]
            predictions = model(inputs.reshape(-1, cfg.n_params).cuda()).cpu()
            all_predictions.append(predictions)

    return torch.cat(all_predictions, dim=0).detach().cpu()

### Compute Test Metrics

In [ ]:
# load the data
manoj_loader = ManojlovicSimulationLoader(specular=False, thick_bottom=True)
physio_params = manoj_loader.load_physiological_parameters()
physical_params = manoj_loader.compute_physical_parameters(physio_params)
reflectances = torch.from_numpy(manoj_loader.load_reflectances())

# sanity check and ensure that there are no duplicates
print(f"Physical params shape: {physical_params.shape}")
print(f"Physical params min: {physical_params.min(dim=0).values.min(dim=0).values}")
print(f"Physical params max: {physical_params.max(dim=0).values.max(dim=0).values}")
unique_params = torch.unique(physical_params, dim=0)
print(f"Unique physical params: {len(unique_params)}")

# use random subset of half the size for training (as in the paper)
n_samples = len(reflectances) // 2
torch.manual_seed(0)
indices = torch.randperm(len(reflectances))[:n_samples]
physical_params_half = physical_params[indices]
reflectances_half = reflectances[indices]
print(f"Physical params shape: {physical_params_half.shape}")
print(f"Reflectances shape: {reflectances_half.shape}")

# load the test data
split = SOTAPreprocessor("manoj", 351, False, None)
ids = split.consistent_data_split_ids(physical_params_half.numpy(), mode="test")
physical_params_test = physical_params_half[ids]
reflectances_test = reflectances_half[ids]
print(f"Physical params test set shape: {physical_params_test.shape}")
print(f"Reflectances test set shape: {reflectances_test.shape}")

# run surrogate model on the test data
manoj_test_pred = manoj_model_forward(physical_params_test, batch_size=1000).reshape(
    -1, len(wavelengths_manoj)
)

# comptue the MAPE, NMAE, and NRMSE
print(f"MAPE: {mape(manoj_test_pred, reflectances_test)}")
print(f"NMAE: {nmae(manoj_test_pred, reflectances_test)}")
print(f"NRMSE: {nrmse(manoj_test_pred, reflectances_test)}")

### Run Forward Model of Physical Parameters for SOTA Comparison

In [ ]:
# sample physical parameters
manoj_loader = ManojlovicSimulationLoader(specular=False, thick_bottom=True)
physio_params = manoj_loader.generate_physiological_parameters(
    DataConfig.DEFAULT_SUBSAMPLE_SIZE_SURROGATE_MODEL, seed=59
)
physical_params = manoj_loader.compute_physical_parameters(physio_params)

# run surrogate model
reflectances_manoj = manoj_model_forward(physical_params).reshape(
    -1, len(wavelengths_manoj)
)
print(f"Reflectances shape: {reflectances_manoj.shape}")

In [ ]:
# plot example reflectance spectra
n_samples = 5
plt.figure(figsize=(10, 5))
for i in range(n_samples):
    plt.plot(
        wavelengths_manoj,
        reflectances_manoj.reshape(-1, len(wavelengths_manoj))[i].numpy(),
        label=f"Sample {i}",
    )
plt.xlabel("Wavelength [nm]")
plt.ylabel("Reflectance")
plt.title("Example Reflectance Spectra (Manojlovic et al. 2025)")
plt.grid()
plt.legend()
plt.show()

In [ ]:
# plot example reflectance spectra in ranges
# depicted in the publication of Manojlovic et al.: 430-750 nm
filter = np.where((wavelengths_manoj >= 430) & (wavelengths_manoj <= 750))[0]

n_samples = 5
plt.figure(figsize=(10, 5))
for i in range(n_samples):
    plt.plot(
        wavelengths_manoj[filter],
        reflectances_manoj.reshape(-1, len(wavelengths_manoj))[i].numpy()[filter],
        label=f"Sample {i}",
    )
plt.xlabel("Wavelength [nm]")
plt.ylabel("Reflectance")
plt.title("Example Reflectance Spectra [430-750 nm] (Manojlovic et al. 2025)")
plt.grid()
plt.legend()
plt.show()

In [ ]:
# save the reflectance spectra for PCA coverage comparison
np.savez_compressed(
    os.path.join(
        RESULT_DIR,
        "reflectances_manoj.npz",
    ),
    reflectances_manoj=reflectances_manoj.numpy(),
)
np.savez_compressed(
    os.path.join(
        RESULT_DIR,
        "physical_params_manoj.npz",
    ),
    physical_params=physical_params.numpy(),
)

In [ ]:
# test loading the reflectance spectra and physical parameters
reflectances_manoj = np.load(
    os.path.join(
        RESULT_DIR,
        "reflectances_manoj.npz",
    )
)["reflectances_manoj"]
physical_params = np.load(
    os.path.join(
        RESULT_DIR,
        "physical_params_manoj.npz",
    )
)["physical_params"]
print(f"Reflectances shape: {reflectances_manoj.shape}")
print(f"Physical params shape: {physical_params.shape}")

## Our Data Surrogate Model Reference Set

### Define Physiological to Physical Parameter Transformation

In [ ]:
wavelengths_issi = torch.from_numpy(
    np.linspace(300, 1000, 351, endpoint=True, dtype=np.float64)
)  # in nm
print(f"Wavelengths: {wavelengths_issi.tolist()}")


def load_issi_model() -> tuple[BaseModel, Any, Any]:
    """
    Load the trained ISSI model.

    :return: The trained ISSI model.
    """
    base_path = os.path.join(
        os.environ["data_dir"], cfg["surrogate"]["issi_model"]["base_path"]
    )
    checkpoint_path = cfg["surrogate"]["issi_model"]["checkpoint_path"]
    return load_trained_model(base_path, checkpoint_path, BaseModel)


def issi_model_forward(
    physical_params: torch.Tensor, batch_size: int = 1000
) -> torch.Tensor:
    """
    Compute the reflectance spectra for the given physical parameters.

    Args:
        physical_params: The physical parameter Tensor to consider.
        batch_size: The surrogate model prediction batch size (controlling GPU load)

    Returns:
        The reflectance spectra.
    """
    # load the model
    model, preprocessor, cfg = load_issi_model()
    # update pre-processor properties for the prediction
    preprocessor.n_wavelengths = physical_params.shape[1]
    # pre-process the physical_params
    data = preprocessor(
        torch.cat(
            [
                physical_params,
                torch.zeros((physical_params.shape[0], physical_params.shape[1], 1)),
            ],
            dim=-1,
        )
    )
    if cfg.dataset.thick_deepest_layer:
        # set thickness of the deepest layer to zero to avoid artifacts
        data[..., -2] = 0

    # Create a TensorDataset and DataLoader
    dataset = TensorDataset(data[..., :-1])
    batch_size = len(data) if batch_size == -1 else batch_size
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    # Run predictions in batches
    all_predictions = []
    model.eval().cuda()
    with torch.no_grad():
        for batch in track(dataloader, description="Predicting reflectance spectra"):
            inputs = batch[0]
            predictions = model(inputs.reshape(-1, cfg.n_params).float().cuda()).cpu()
            all_predictions.append(predictions)

    return torch.cat(all_predictions, dim=0).detach().cpu()

### Sample Physical Parameters

In [ ]:
# load physical parameters
physical_params = torch.from_numpy(GenericSimulationLoader().load_physical_parameters())
print(f"Physical params shape: {physical_params.shape}")

# random subsample to match the amount of spectra with the other models
physical_params = subsample_data(
    physical_params, DataConfig.DEFAULT_SUBSAMPLE_SIZE_SURROGATE_MODEL
)

# check the physical parameter ranges
print(f"Physical params shape: {physical_params.shape}")
print(f"Physical params min: {physical_params.min(dim=0).values.min(dim=0).values}")
print(f"Physical params max: {physical_params.max(dim=0).values.max(dim=0).values}")
# check that there are no duplicates
unique_params = torch.unique(physical_params, dim=0)
print(f"Unique params: {len(unique_params)}")

### Run Forward Model of Physical Parameters for SOTA Comparison

In [ ]:
reflectances_issi = issi_model_forward(physical_params).reshape(
    -1, len(wavelengths_issi)
)
print(f"Reflectances shape: {reflectances_issi.shape}")

In [ ]:
# plot example reflectance spectra
n_samples = 5
plt.figure(figsize=(10, 5))
for i in range(n_samples):
    plt.plot(
        wavelengths_issi,
        reflectances_issi.reshape(-1, len(wavelengths_issi))[i].numpy(),
        label=f"Sample {i}",
    )
plt.xlabel("Wavelength [nm]")
plt.ylabel("Reflectance")
plt.title("Example Reflectance Spectra")
plt.grid()
plt.legend()
plt.show()

In [ ]:
# save the reflectance spectra for PCA coverage comparison
np.savez_compressed(
    os.path.join(
        RESULT_DIR,
        "reflectances_issi.npz",
    ),
    reflectances_issi=reflectances_issi.numpy(),
)
np.savez_compressed(
    os.path.join(
        RESULT_DIR,
        "physical_params_issi.npz",
    ),
    physical_params=physical_params.clone().numpy(),
)

In [ ]:
# test loading the reflectance spectra and physical parameters
reflectances_issi_reloaded = np.load(
    os.path.join(
        RESULT_DIR,
        "reflectances_issi.npz",
    )
)["reflectances_issi"]
physical_params_reloaded = np.load(
    os.path.join(
        RESULT_DIR,
        "physical_params_issi.npz",
    )
)["physical_params"]
print(f"Reflectances shape: {reflectances_issi_reloaded.shape}")
print(f"Physical params shape: {physical_params_reloaded.shape}")

### Run Forward Training Set Size Ablation Models

In [ ]:
def issi_ablation_model_forward(
    physical_params: torch.Tensor, model_name: str, batch_size: int = 1000
) -> torch.Tensor:
    """
    Compute the reflectance spectra for the given physical parameters.

    Args:
        physical_params: The physical parameter Tensor to consider.
        model_name: The name of the ablation model to use ("8400k", "420k", or "21k").
        batch_size: The surrogate model prediction batch size (controlling GPU load)

    Returns:
        The reflectance spectra.
    """
    # Load the model(s)
    if model_name == "8400k":  # 1.0 tdr
        folder_path = "version_0_100M_photons_fold_0_tdr_1.0"
        checkpoint_path = (
            "FCPhysicalForwardSurrogateModel-epoch=499-val_loss=0.0001.ckpt"
        )
    elif model_name == "420k":  # 0.05 tdr
        folder_path = "version_19_100M_photons_fold_0_tdr_0.05"
        checkpoint_path = (
            "FCPhysicalForwardSurrogateModel-epoch=499-val_loss=0.0005.ckpt"
        )
    elif model_name == "21k":  # 0.0025 tdr - same training dataset size as Tsui et al.
        folder_path = "version_50_100M_photons_fold_0_tdr_0.0025/"
        checkpoint_path = "ForwardSurrogateModel-epoch=499-val_loss=0.0067.ckpt"
    else:
        raise ValueError(f"Unknown subset name: {model_name}")
    # Create the model paths
    base_path = os.path.join(os.environ["data_dir"], "models/scaling_models")
    model_dir_path = os.path.join(base_path, folder_path)
    model_ckpt_path = f"checkpoints/{checkpoint_path}"
    model, preprocessor, cfg = load_trained_model(
        model_dir_path, model_ckpt_path, BaseModel
    )
    # Update pre-processor properties for the prediction
    preprocessor.n_wavelengths = physical_params.shape[1]
    # Pre-process the physical_params
    data = preprocessor(
        torch.cat(
            [
                physical_params,
                torch.zeros((physical_params.shape[0], physical_params.shape[1], 1)),
            ],
            dim=-1,
        )
    )
    if cfg.dataset.thick_deepest_layer:
        # Set thickness of the deepest layer to zero to avoid artifacts
        data[..., -2] = 0

    # Create a TensorDataset and DataLoader
    dataset = TensorDataset(data[..., :-1])
    batch_size = len(data) if batch_size == -1 else batch_size
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    # Run predictions in batches
    all_predictions = []
    model.eval().cuda()
    with torch.no_grad():
        for batch in track(dataloader, description="Predicting reflectance spectra"):
            inputs = batch[0]
            predictions = model(inputs.reshape(-1, cfg.n_params).float().cuda()).cpu()
            all_predictions.append(predictions)

    return torch.cat(all_predictions, dim=0).detach().cpu()

In [ ]:
# Repeat inference for specific ablation models
for ablation_model_name in ["8400k", "420k", "21k"]:
    reflectances_issi = issi_ablation_model_forward(
        physical_params, ablation_model_name
    ).reshape(-1, len(wavelengths_issi))
    print(f"Reflectances shape ({ablation_model_name}): {reflectances_issi.shape}")

    # Plot example reflectance spectra
    n_samples = 5
    plt.figure(figsize=(10, 5))
    for i in range(n_samples):
        plt.plot(
            wavelengths_issi,
            reflectances_issi.reshape(-1, len(wavelengths_issi))[i].numpy(),
            label=f"Sample {i}",
        )
    plt.xlabel("Wavelength [nm]")
    plt.ylabel("Reflectance")
    plt.title("Example Reflectance Spectra")
    plt.grid()
    plt.legend()
    plt.show()

    # Save the reflectance spectra for PCA coverage comparison
    np.savez_compressed(
        os.path.join(
            RESULT_DIR,
            f"reflectances_issi_{ablation_model_name}.npz",
        ),
        reflectances_issi=reflectances_issi.numpy(),
    )
    np.savez_compressed(
        os.path.join(
            RESULT_DIR,
            f"physical_params_issi_{ablation_model_name}.npz",
        ),
        physical_params=physical_params.clone().numpy(),
    )

    # Test loading the reflectance spectra and physical parameters
    reflectances_issi_reloaded = np.load(
        os.path.join(
            RESULT_DIR,
            f"reflectances_issi_{ablation_model_name}.npz",
        )
    )["reflectances_issi"]
    physical_params_reloaded = np.load(
        os.path.join(
            RESULT_DIR,
            f"physical_params_issi_{ablation_model_name}.npz",
        )
    )["physical_params"]
    print(f"Reflectances shape: {reflectances_issi_reloaded.shape}")
    print(f"Physical params shape: {physical_params_reloaded.shape}")